In [2]:
import os
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

import cv2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
DATA_PATH = "../Assignment-3/Data/Clothes_Dataset"
IMG_SIZE = 64

X = []
y = []
classes = os.listdir(DATA_PATH)

for label in classes:
    folder = os.path.join(DATA_PATH, label)
    for img_name in tqdm(os.listdir(folder), desc=f"Loading {label}"):
        img_path = os.path.join(folder, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img.astype(np.float32) / 255.0
        X.append(img)
        y.append(label)

X = np.array(X)
y = np.array(y)

# Normalize images (standardization)
mean = X.mean(axis=(0, 1, 2), keepdims=True)
std = X.std(axis=(0, 1, 2), keepdims=True)
X = (X - mean) / (std + 1e-7)

print(X.shape, y.shape)


Loading Sweter: 100%|██████████| 500/500 [00:03<00:00, 127.69it/s]


(7500, 64, 64, 3) (7500,)


In [ ]:
from sklearn.model_selection import train_test_split

le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)


In [ ]:
class ClothesImageDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.tensor(images).permute(0,3,1,2)  # NCHW
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

train_ds = ClothesImageDataset(X_train, y_train)
test_ds  = ClothesImageDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)


In [ ]:
class CNN3(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Flatten(),
            nn.Linear(128*8*8, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)


class CNN2(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Flatten(),
            nn.Linear(64*16*16, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

class CNN1(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            nn.Flatten(),
            nn.Linear(32*32*32, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
def train(model, optimizer, criterion, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

def evaluate(model):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            out = model(xb)
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(yb.numpy())
    return f1_score(trues, preds, average="macro")


In [ ]:
results = []

configs = [
    ("CNN-3", CNN3),
    ("CNN-2", CNN2),
    ("CNN-1", CNN1),
]

for name, model_class in configs:
    print(f"\nTraining: {name}")
    model = model_class(num_classes).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    train(model, optimizer, criterion, epochs=30)
    f1 = evaluate(model)

    results.append((name, f1))
    print(f"{name} | Macro F1: {f1:.4f}")



Training: CNN-3
  Epoch 5/30, Loss: 1.1617
  Epoch 10/30, Loss: 0.7386
  Epoch 15/30, Loss: 0.4539
  Epoch 20/30, Loss: 0.3288
  Epoch 25/30, Loss: 0.2495
  Epoch 30/30, Loss: 0.2027
CNN-3 | Macro F1: 0.6271

Training: CNN-2
  Epoch 5/30, Loss: 1.1450
  Epoch 10/30, Loss: 0.5809
  Epoch 15/30, Loss: 0.3404
  Epoch 20/30, Loss: 0.2549
  Epoch 25/30, Loss: 0.1986
  Epoch 30/30, Loss: 0.1559
CNN-2 | Macro F1: 0.5516

Training: CNN-1
  Epoch 5/30, Loss: 1.0000
  Epoch 10/30, Loss: 0.3639
  Epoch 15/30, Loss: 0.2260
  Epoch 20/30, Loss: 0.1789
  Epoch 25/30, Loss: 0.1643
  Epoch 30/30, Loss: 0.1154
CNN-1 | Macro F1: 0.4665


In [ ]:
import pandas as pd
df = pd.DataFrame(results, columns=["Model","Macro F1"])
df


,Model,Macro F1
0,CNN-3,0.627072
1,CNN-2,0.551629
2,CNN-1,0.466507
